# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/si-ux/FlyrankAI-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> **Lane:** Ranking Signal Analysis · **Dataset:** the full warehouse release
> (`FlyRank/internship-warehouse`, build v20260703, ~79M daily rows).
>
> This notebook is the *contract*: what one row means, which fields may be used for what, and
> over which windows. Every claim below has an executed query under it. Heavy scans are cached
> to `work/outputs/` so a rerun is cheap — delete that folder to force a fresh scan.

### Setup

Paste your Hugging Face **READ** token at the prompt, or set `HF_TOKEN` in Colab's Secrets
panel (🔑). Never type a token into a cell — this repo is public.

In [1]:
%pip install -q duckdb huggingface_hub pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, sys, json, time, getpass, pathlib
import duckdb, pandas as pd

# --- token: env -> Colab secrets -> local HF cache -> prompt. Never hardcoded. ---
def _token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    try:
        from google.colab import userdata
        t = userdata.get("HF_TOKEN")
        if t:
            return t
    except Exception:
        pass
    cached = pathlib.Path.home() / ".cache/huggingface/token"
    if cached.exists() and cached.read_text().strip():
        return cached.read_text().strip()
    return getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")   # first run downloads the extension (~a few min)
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [_token()])

B   = "hf://datasets/FlyRank/internship-warehouse"
CLI = f"read_parquet('{B}/dim_clients.parquet')"
CNT = f"read_parquet('{B}/dim_content.parquet')"
QRY = f"read_parquet('{B}/fact_content_query_90d.parquet')"
def FACT(*months):
    """Explicit list of month partitions — hf:// does not support {brace} globs."""
    return "read_parquet([" + ",".join(
        f"'{B}/fact_content_daily_performance/month={m}/*.parquet'" for m in months) + "])"

# --- cache: heavy scans run once, then reload from work/outputs/ ---
REPO = pathlib.Path.cwd()
for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (p / "data/raw/content_refresh_anonymized.csv").exists():
        REPO = p
        break
OUT = REPO / "work/outputs"
OUT.mkdir(parents=True, exist_ok=True)
rel = lambda q: q.relative_to(REPO).as_posix()   # keep local paths out of committed output

def cached(name, sql):
    """Run sql once; cache the (small) result frame as parquet next to the report."""
    f = OUT / f"w03_{name}.parquet"
    if f.exists():
        return pd.read_parquet(f)
    t = time.time()
    df = con.sql(sql).df()
    df.to_parquet(f, index=False)
    print(f"[scanned {name} in {time.time()-t:.0f}s -> cached]")
    return df

print("connected · cache dir:", rel(OUT))

connected · cache dir: work/outputs


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one pseudonymized content item (a page), owned by one client, evaluated once.**

The warehouse's `fact_content_daily_performance` is at `report_date × client × content` grain —
one row per page per day. That is *not* my unit of analysis. I collapse it to one row per
content item by aggregating over two **disjoint** windows:

| Window | Dates | Role |
|---|---|---|
| Feature window | 2026-01-01 → 2026-03-31 (90 days) | everything the model may look at |
| Outcome window | 2026-04-01 → 2026-04-30 (30 days) | where the label is measured |
| **Sealed test** | features 2026-03-01 → 2026-05-31, label **2026-06** | touched once, at the very end |

The two development windows do not overlap by a single day, so every feature is knowable at
the moment of prediction (2026-03-31) and the label is strictly after it. The panel runs
2025-01-27 → 2026-06-30; I develop on a **mid-panel** split and keep the final month (June 2026)
sealed, because developing label logic on the last month means developing inside my own test
window.

Position is **impression-weighted**, never a mean of daily averages:
`SUM(gsc_sum_position) / SUM(gsc_impressions)`. Averaging daily `gsc_avg_position` would weight
a 2-impression day the same as a 20,000-impression day.

In [3]:
# CLAIM: the daily fact's grain is report_date x client x content -> the probe returns 0 rows.
probe = cached("grain_probe", f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT('2026-03')}
    GROUP BY 1,2,3 HAVING COUNT(*) > 1 LIMIT 5
""")
print(f"Duplicate (date, client, content) keys: {len(probe)}  -> grain holds" if probe.empty
      else probe)

# CLAIM: the panel spans 2025-01-27 .. 2026-06-30, and my two windows sit inside it, disjoint.
span = cached("panel_span", f"""
    SELECT MIN(report_date) AS panel_start, MAX(report_date) AS panel_end
    FROM {FACT('2025-01','2026-06')}
""")
print(span.to_string(index=False))

Duplicate (date, client, content) keys: 0  -> grain holds
panel_start  panel_end
 2025-01-27 2026-06-30


In [4]:
# CLAIM: the two windows are disjoint and each is fully populated.
shapes = cached("month_shapes", f"""
    SELECT month,
           COUNT(*)                          AS n_rows,
           COUNT(DISTINCT client_hash_id)    AS n_clients,
           COUNT(DISTINCT content_hash_id)   AS n_content,
           SUM(gsc_impressions)              AS impressions
    FROM {FACT('2026-01','2026-02','2026-03','2026-04','2026-05','2026-06')}
    GROUP BY month ORDER BY month
""")
print(shapes.to_string(index=False))
print("\nFeature window = 2026-01..2026-03   Outcome = 2026-04   Sealed = 2026-06")

  month   n_rows  n_clients  n_content  impressions
2026-01  7890817         42     261984  144354689.0
2026-02  7355108         54     321546  180128922.0
2026-03  9841378         55     331437  280657589.0
2026-04 10424730         61     362172  292067218.0
2026-05 11687376         66     389153  266164899.0
2026-06 11694072         65     409205  216194872.0

Feature window = 2026-01..2026-03   Outcome = 2026-04   Sealed = 2026-06


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Every field I may touch, in exactly one bucket. The rule that decides the hard cases: *could I
have known this at 2026-03-31?*

**FEATURE** — measured inside the feature window only (Jan–Mar 2026), one row per content item:

| Field | Source | Why it is safe |
|---|---|---|
| `f_pos` (impression-weighted avg position) | `fact` Jan–Mar | the ranking state I am predicting *from* |
| `f_impressions`, `f_clicks` | `fact` Jan–Mar | demand + capture, both fully in-window |
| `f_ctr` = clicks/impressions | derived | ratio of two in-window totals |
| `f_days_with_impressions` | `fact` Jan–Mar | consistency of visibility |
| `f_pos_volatility` (stddev of daily position) | `fact` Jan–Mar | instability is the signal of the lane |
| `f_pos_trend` (Mar position − Jan position) | `fact` Jan–Mar | direction *before* the cut, not across it |
| `content_type`, `main_intent`, `competition_level` | `dim_content` | static metadata |
| `search_volume`, `competition`, `cpc`, `backlinks` | `dim_content` | keyword context, not outcome-derived |
| `word_count`, `char_count`, `keyword_token_count` | `dim_content` | content properties |
| `content_age_days` at 2026-03-31 | derived from `content_created_date` | computed *at the cut date*, not today |

**LABEL** — measured in the outcome window only (April 2026):

| Field | Definition |
|---|---|
| `o_pos` | impression-weighted avg position, April 2026 |
| `pos_delta` | `o_pos − f_pos` (positive = ranking got worse) |
| **`is_position_decline`** | **1 when `pos_delta ≥ 1.0`** — the target |

**CONTEXT** — for grouping, joining, splitting, reading. Never model inputs:

`client_hash_id` (the group for the client-holdout split), `content_hash_id`, `url_hash_id`,
`keyword_hash_id`, `report_date`, `month`.

**EXCLUDED** — with a one-line why each:

| Field | Why excluded |
|---|---|
| `o_pos`, `pos_delta`, anything from April 2026 | the label and its siblings — using them is the label leak |
| **every column of `fact_content_query_90d`** | its window is **2026-04-02 → 2026-06-30**, which *starts inside my outcome month*. Even `*_prev30` sits after my feature cut. It is future information for this label — verified below, not assumed |
| `gsc_avg_position` averaged per-day | not wrong but misleading — unweighted; I use the impression-weighted form |
| `content_hash_id` / `client_hash_id` as features | pseudonyms; using them lets the model memorize a client instead of learning a signal |
| `is_deleted`, `is_published` | platform state that can change after the cut |
| `last_optimized_date`, `optimization_eligible_date` | **decision-derived** — they encode an action FlyRank's own system already chose to take. Learning them means learning the old rule, not the world |
| GA4 columns (`ga4_*`, `sessions_*`, `ai_*`, `scroll_events`) | in-scope in principle, but coverage is three-valued and thin (see limits) — I exclude them from the core model and note it as a deliberate narrowing |

In [5]:
# CLAIM: the query table's window starts inside my outcome month -> it is future information.
w = cached("query_window", f"""
    SELECT MIN(window_start) AS window_start, MAX(window_end) AS window_end,
           COUNT(*) AS n_rows
    FROM {QRY}
""")
print(w.to_string(index=False))

ws = pd.to_datetime(w.window_start.iloc[0])
print(f"\nFeature cut : 2026-03-31")
print(f"Outcome     : 2026-04-01 .. 2026-04-30")
print(f"Query window: {ws.date()} .. {pd.to_datetime(w.window_end.iloc[0]).date()}")
assert ws > pd.Timestamp("2026-03-31"), "query window would need re-checking"
print("\nVERDICT: query-table window starts AFTER the feature cut and inside the outcome "
      "month -> EXCLUDED from features for this label. Not a judgement call; a date check.")

window_start window_end  n_rows
  2026-04-02 2026-06-30 2414248

Feature cut : 2026-03-31
Outcome     : 2026-04-01 .. 2026-04-30
Query window: 2026-04-02 .. 2026-06-30

VERDICT: query-table window starts AFTER the feature cut and inside the outcome month -> EXCLUDED from features for this label. Not a judgement call; a date check.


In [6]:
# The contract as executable lists. Anything not in a bucket is not allowed near the model.
FEATURES = ["f_pos","f_impressions","f_clicks","f_ctr","f_days_with_impressions",
            "f_pos_volatility","f_pos_trend","content_type","main_intent","competition_level",
            "search_volume","competition","cpc","backlinks","word_count","char_count",
            "keyword_token_count","content_age_days"]
LABEL    = ["o_pos","pos_delta","is_position_decline"]
CONTEXT  = ["client_hash_id","content_hash_id","url_hash_id","keyword_hash_id","report_date","month"]
EXCLUDED = ["last_optimized_date","optimization_eligible_date","is_deleted","is_published",
            "gsc_avg_position"] + [f"query90d.{c}" for c in
            ["impressions_90d","clicks_90d","impressions_last30","clicks_last30",
             "impressions_prev30","clicks_prev30","avg_position_90d","avg_position_last30",
             "avg_position_prev30","content_total_impressions_90d"]]

overlap = (set(FEATURES) & set(LABEL)) | (set(FEATURES) & set(CONTEXT)) | (set(FEATURES) & set(EXCLUDED))
assert not overlap, f"a field is in two buckets: {overlap}"
print(f"FEATURES {len(FEATURES)} | LABEL {len(LABEL)} | CONTEXT {len(CONTEXT)} | EXCLUDED {len(EXCLUDED)}")
print("No field appears in two buckets.")

json.dump({"features":FEATURES,"label":LABEL,"context":CONTEXT,"excluded":EXCLUDED,
           "feature_window":["2026-01-01","2026-03-31"],
           "outcome_window":["2026-04-01","2026-04-30"],
           "sealed_test_label_month":"2026-06",
           "label_rule":"is_position_decline = (o_pos - f_pos) >= 1.0",
           "population":"f_impressions >= 100 AND o_impressions >= 30"},
          open(OUT/"w03_data_contract.json","w"), indent=2)
print("contract receipt ->", rel(OUT/"w03_data_contract.json"))

FEATURES 18 | LABEL 3 | CONTEXT 6 | EXCLUDED 15
No field appears in two buckets.
contract receipt -> work/outputs/w03_data_contract.json


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
# CLAIM: the published table counts are what I actually see.
counts = cached("table_counts", f"""
    SELECT 'dim_clients' AS tbl, COUNT(*) AS n FROM {CLI}
    UNION ALL SELECT 'dim_content', COUNT(*) FROM {CNT}
    UNION ALL SELECT 'fact_content_query_90d', COUNT(*) FROM {QRY}
""")
expected = {"dim_clients":104, "dim_content":519606, "fact_content_query_90d":2414248}
counts["published"] = counts.tbl.map(expected)
counts["matches"]   = counts.n == counts.published
print(counts.to_string(index=False))
assert counts.matches.all(), "row counts drifted from the documented build"

                   tbl       n  published  matches
           dim_clients     104        104     True
           dim_content  519606     519606     True
fact_content_query_90d 2414248    2414248     True


In [8]:
# CLAIM: access flags are THREE-valued. `= FALSE` silently drops the NULL clients.
flags = cached("client_flags", f"""
    SELECT has_gsc_access, has_ga4_access, COUNT(*) AS n_clients
    FROM {CLI} GROUP BY 1,2 ORDER BY n_clients DESC
""")
print(flags.to_string(index=False))

n_null = int(flags[flags.has_gsc_access.isna()].n_clients.sum())
print(f"\nClients with NULL has_gsc_access: {n_null}")
print("-> filter with IS TRUE / IS NOT TRUE, never = TRUE / = FALSE.")

avail = cached("fact_flags", f"""
    SELECT gsc_data_available, ga4_data_available, COUNT(*) AS n_rows
    FROM {FACT('2026-03')} GROUP BY 1,2 ORDER BY n_rows DESC
""")
print("\nDaily-fact availability flags, March 2026:")
print(avail.to_string(index=False))
ga4_null = int(avail[avail.ga4_data_available.isna()].n_rows.sum())
print(f"\nRows with NULL ga4_data_available in ONE month: {ga4_null:,}"
      f"  ({ga4_null/avail.n_rows.sum():.1%}) -> GA4 is not safely usable as a dense feature.")

 has_gsc_access  has_ga4_access  n_clients
           True            True         53
          False           False         26
           True           False         14
           <NA>            <NA>         10
          False            True          1

Clients with NULL has_gsc_access: 10
-> filter with IS TRUE / IS NOT TRUE, never = TRUE / = FALSE.

Daily-fact availability flags, March 2026:
 gsc_data_available  ga4_data_available  n_rows
              False               False 4690323
               True               False 1718348
               True                <NA> 1528366
              False                <NA> 1490375
               True                True  364347
              False                True   49619

Rows with NULL ga4_data_available in ONE month: 3,018,741  (30.7%) -> GA4 is not safely usable as a dense feature.


In [9]:
# CLAIM: the population and label rule produce the size and base rate the contract states.
label = cached("label_dist", f"""
    WITH feat AS (
      SELECT content_hash_id, client_hash_id,
             SUM(gsc_impressions)                                        AS f_imps,
             SUM(gsc_sum_position)/NULLIF(SUM(gsc_impressions),0)        AS f_pos
      FROM {FACT('2026-01','2026-02','2026-03')}
      WHERE gsc_data_available IS TRUE GROUP BY 1,2),
    outc AS (
      SELECT content_hash_id,
             SUM(gsc_impressions)                                        AS o_imps,
             SUM(gsc_sum_position)/NULLIF(SUM(gsc_impressions),0)        AS o_pos
      FROM {FACT('2026-04')}
      WHERE gsc_data_available IS TRUE GROUP BY 1)
    SELECT COUNT(*)                                                      AS eligible_items,
           COUNT(DISTINCT f.client_hash_id)                              AS n_clients,
           AVG(CASE WHEN o.o_pos - f.f_pos >= 1.0 THEN 1.0 ELSE 0 END)   AS decline_base_rate,
           MEDIAN(f.f_pos)                                               AS median_feature_pos,
           MEDIAN(o.o_pos - f.f_pos)                                     AS median_pos_delta
    FROM feat f JOIN outc o USING (content_hash_id)
    WHERE f.f_imps >= 100 AND o.o_imps >= 30
""")
print(label.to_string(index=False))
r = label.iloc[0]
print(f"\nUnit of analysis: {int(r.eligible_items):,} content items across "
      f"{int(r.n_clients)} clients.")
print(f"Base rate to beat: {r.decline_base_rate:.1%} — always print this next to any score.")

 eligible_items  n_clients  decline_base_rate  median_feature_pos  median_pos_delta
         106461         42           0.567259             8.17527          1.529715

Unit of analysis: 106,461 content items across 42 clients.
Base rate to beat: 56.7% — always print this next to any score.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

What this data can **never** tell me:

1. **No causality.** The panel is observational. I can say a page's position worsened after a
   given signal state; I cannot say the signal *caused* it, and I can never attribute a move to
   a Google algorithm update. There is no randomisation and no control group.

2. **Portfolio-wide drift contaminates the label.** The median position change Jan–Mar → April
   is **+1.53 positions** — the *typical* page drifts down. So `is_position_decline` partly
   encodes a market-wide movement, not page-specific decay. A model can score well by learning
   "everything drifts" — which is why the base rate must sit next to every metric, and why
   per-client normalisation is worth testing in ML-08.

3. **Unbalanced panel.** Only 37 of the clients present in 2026 appear in all six months;
   9 appear in just two. Content coverage grows from 262k items (January) to 409k (June) —
   pages *enter* the panel constantly. A page absent in January is not a page with zero
   impressions in January; it is a page that did not exist in the export yet.

4. **Population selection uses outcome-window information.** Eligibility requires ≥ 30
   impressions in **April** — the outcome month. That is deliberate (position on 3 impressions
   is noise), but it means the population is conditioned on surviving into the label window.
   Pages that vanished entirely are excluded, so this measures *decline among pages that stayed
   measurable*, not decline including disappearance. Disclosed, not hidden.

5. **GA4 is not dense enough to model on.** In March 2026 alone, ~3.0M rows carry a **NULL**
   `ga4_data_available` — neither zero-filled nor flagged false. Engagement, scroll, and AI
   traffic are therefore excluded from the core model rather than imputed.

6. **The query table is unusable for this label.** Its fixed window (2026-04-02 → 2026-06-30)
   begins inside my outcome month. Query-mix diversity would have been a strong lane feature;
   the window forbids it. Recovering it would require moving the label to a period after
   2026-06-30 — which the panel does not have.

7. **Position ≠ traffic ≠ revenue.** `gsc_avg_position` is a search-console average over an
   opaque set of queries. A page can improve position on low-value queries and lose clicks.
   Nothing here measures revenue, conversion, or business value.

In [10]:
# CLAIM: unbalanced panel — most clients do NOT span the full window.
depth = cached("client_depth", f"""
    WITH d AS (
      SELECT client_hash_id, COUNT(DISTINCT month) AS months_present
      FROM {FACT('2026-01','2026-02','2026-03','2026-04','2026-05','2026-06')}
      GROUP BY 1)
    SELECT months_present, COUNT(*) AS n_clients
    FROM d GROUP BY 1 ORDER BY months_present
""")
print(depth.to_string(index=False))
full = int(depth[depth.months_present==6].n_clients.iloc[0])
print(f"\n{full} of {int(depth.n_clients.sum())} clients span all six months "
      f"-> per-client windows beat one global calendar window.")

# CLAIM: history start dates are missing for a large minority of clients.
starts = cached("client_starts", f"""
    SELECT COUNT(*) AS n_clients,
           COUNT(*) FILTER (WHERE gsc_data_start IS NULL) AS null_gsc_start,
           MIN(gsc_data_start) AS earliest, MAX(gsc_data_start) AS latest
    FROM {CLI}
""")
print("\n" + starts.to_string(index=False))
print("-> gsc_data_start is NULL for a large minority; it cannot be relied on alone "
      "to define per-client windows.")

 months_present  n_clients
              2          9
              3          6
              4          5
              5         13
              6         37

37 of 70 clients span all six months -> per-client windows beat one global calendar window.

 n_clients  null_gsc_start   earliest     latest
       104              37 2025-01-27 2026-06-02
-> gsc_data_start is NULL for a large minority; it cannot be relied on alone to define per-client windows.


### Output — what this hands to a human

**One sentence:** for each measurable content item, a ranked, reason-coded estimate of how
likely its average search position is to deteriorate by a full position or more over the next
30 days — so an SEO team can order its review queue by observed risk rather than by intuition.

Framed as **decision-support**, on **observed** and **measured** signals, and **directional**
about what tends to precede a decline. Never causal, never a claim about Google's algorithm.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.